# Backpropagation from Scratch: Electrical Signal Classification

สร้าง Neural Network ด้วย **NumPy ล้วน ๆ** เพื่อจำแนกรูปคลื่นไฟฟ้า 4 ชนิด: **Sine, Square, Triangle และ Sawtooth**

แต่ละสัญญาณถูกสุ่ม amplitude, frequency, phase, DC offset และ noise เพื่อให้โมเดลเรียนรู้ลักษณะรูปคลื่น ไม่ใช่จำตัวอย่าง

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
CLASS_NAMES = np.array(['Sine', 'Square', 'Triangle', 'Sawtooth'])
CLASS_COLORS = ['#118ab2', '#ef476f', '#06d6a0', '#ff9f1c']
print('Electrical signal classifier ready')

## 1) สร้างชุดข้อมูลสัญญาณไฟฟ้า
กำหนด sampling rate = 128 Hz และใช้ข้อมูล 1 วินาทีต่อหนึ่งตัวอย่าง (128 samples)

In [ ]:
FS = 128
N_SAMPLES = 128
TIME = np.arange(N_SAMPLES) / FS

def make_wave(class_id, rng):
    amplitude = rng.uniform(0.7, 1.3)
    frequency = rng.choice([3.0, 4.0, 5.0])
    phase = rng.uniform(-0.15, 0.15)
    dc_offset = rng.uniform(-0.15, 0.15)
    noise_level = rng.uniform(0.03, 0.12)
    angle = 2*np.pi*frequency*TIME + phase
    cycle = (frequency*TIME + phase/(2*np.pi)) % 1.0
    if class_id == 0:
        clean = np.sin(angle)
    elif class_id == 1:
        clean = np.where(np.sin(angle) >= 0, 1.0, -1.0)
    elif class_id == 2:
        clean = 2.0 * np.abs(2.0*cycle - 1.0) - 1.0
    else:
        clean = 2.0*cycle - 1.0
    signal = amplitude*clean + dc_offset + rng.normal(0, noise_level, N_SAMPLES)
    return signal

def make_dataset(per_class=400, seed=123):
    rng = np.random.default_rng(seed)
    X, y = [], []
    for class_id in range(4):
        for _ in range(per_class):
            X.append(make_wave(class_id, rng)); y.append(class_id)
    order = rng.permutation(len(X))
    return np.asarray(X)[order], np.asarray(y)[order]

X, y = make_dataset()
fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True)
for class_id, ax in enumerate(axes.flat):
    example = X[np.where(y == class_id)[0][0]]
    ax.plot(TIME, example, color=CLASS_COLORS[class_id], lw=2)
    ax.set_title(CLASS_NAMES[class_id]); ax.set_ylabel('Voltage (normalized)'); ax.grid(alpha=.25)
axes[-1,0].set_xlabel('Time (s)'); axes[-1,1].set_xlabel('Time (s)')
plt.tight_layout(); plt.show()
print('Dataset:', X.shape, '| Targets:', y.shape)

## 2) แบ่ง Training และ Testing
ใช้ Training 80% และเก็บ Testing 20% ไว้วัดกับข้อมูลที่โมเดลไม่เคยเห็น จากนั้น normalize แต่ละ waveform เพื่อลดผลของ amplitude และ DC offset

In [ ]:
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

def normalize_each_wave(X):
    mean = X.mean(axis=1, keepdims=True)
    std = X.std(axis=1, keepdims=True) + 1e-8
    return (X - mean) / std

X_train = normalize_each_wave(X_train)
X_test = normalize_each_wave(X_test)
print('Train:', X_train.shape, '| Test:', X_test.shape)

## 3) Neural Network: 128 → 64 → 4

- Input: ค่าแรงดัน 128 samples
- Hidden layer: 64 neurons + ReLU
- Output: 4 neurons + Softmax
- Loss: Cross-Entropy

In [ ]:
def initialize(seed=7):
    rng = np.random.default_rng(seed)
    return {
        'W1': rng.normal(0, np.sqrt(2/128), (128, 64)),
        'b1': np.zeros((1, 64)),
        'W2': rng.normal(0, np.sqrt(2/64), (64, 4)),
        'b2': np.zeros((1, 4))
    }

def softmax(z):
    exp_z = np.exp(z - z.max(axis=1, keepdims=True))
    return exp_z / exp_z.sum(axis=1, keepdims=True)

def forward(X, p):
    Z1 = X @ p['W1'] + p['b1']
    A1 = np.maximum(0, Z1)
    Z2 = A1 @ p['W2'] + p['b2']
    P = softmax(Z2)
    return P, {'X': X, 'Z1': Z1, 'A1': A1, 'P': P}

def loss_fn(P, y):
    return -np.mean(np.log(P[np.arange(len(y)), y] + 1e-12))

def backward(y, p, cache):
    n = len(y)
    dZ2 = cache['P'].copy()
    dZ2[np.arange(n), y] -= 1
    dZ2 /= n
    dW2 = cache['A1'].T @ dZ2
    db2 = dZ2.sum(axis=0, keepdims=True)
    dA1 = dZ2 @ p['W2'].T
    dZ1 = dA1 * (cache['Z1'] > 0)
    dW1 = cache['X'].T @ dZ1
    db1 = dZ1.sum(axis=0, keepdims=True)
    return {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}

def evaluate(X, y, p):
    P, _ = forward(X, p)
    return loss_fn(P, y), np.mean(P.argmax(axis=1) == y)

## 4) ตรวจ Backpropagation ด้วย Numerical Gradient
ถ้า relative error เล็กมาก แสดงว่า gradient จาก Backpropagation ตรงกับค่าประมาณเชิงตัวเลข

In [ ]:
params = initialize()
small_X, small_y = X_train[:20], y_train[:20]
P, cache = forward(small_X, params)
analytical = backward(small_y, params, cache)['W2'][0, 0]
eps = 1e-5
original = params['W2'][0, 0]
params['W2'][0, 0] = original + eps
plus = loss_fn(forward(small_X, params)[0], small_y)
params['W2'][0, 0] = original - eps
minus = loss_fn(forward(small_X, params)[0], small_y)
params['W2'][0, 0] = original
numerical = (plus - minus) / (2*eps)
error = abs(analytical-numerical)/(abs(analytical)+abs(numerical)+1e-12)
print(f'Backprop gradient : {analytical:.8f}')
print(f'Numerical gradient: {numerical:.8f}')
print(f'Relative error    : {error:.2e}', 'PASS' if error < 1e-6 else 'CHECK')

## 5) Training ด้วย Mini-batch Gradient Descent
ทุก batch จะทำ `Forward → Loss → Backward → Update weights`

In [ ]:
def train(X, y, X_test, y_test, epochs=80, lr=0.03, batch_size=64):
    p = initialize()
    history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}
    rng = np.random.default_rng(99)
    for epoch in range(epochs):
        order = rng.permutation(len(X))
        for start in range(0, len(X), batch_size):
            idx = order[start:start+batch_size]
            P, cache = forward(X[idx], p)
            grads = backward(y[idx], p, cache)
            for key in p:
                p[key] -= lr * grads[key]
        tr_loss, tr_acc = evaluate(X, y, p)
        te_loss, te_acc = evaluate(X_test, y_test, p)
        history['train_loss'].append(tr_loss); history['test_loss'].append(te_loss)
        history['train_acc'].append(tr_acc); history['test_acc'].append(te_acc)
        if epoch % 10 == 0 or epoch == epochs-1:
            print(f'Epoch {epoch+1:3d} | loss={tr_loss:.4f} | train={tr_acc:.1%} | test={te_acc:.1%}')
    return p, history

params, history = train(X_train, y_train, X_test, y_test)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history['train_loss'], label='Training'); ax[0].plot(history['test_loss'], label='Testing')
ax[0].set(title='Loss', xlabel='Epoch', ylabel='Cross-entropy'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(np.array(history['train_acc'])*100, label='Training')
ax[1].plot(np.array(history['test_acc'])*100, label='Testing')
ax[1].set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy (%)', ylim=(20, 102)); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 6) Confusion Matrix บน Testing Data

In [ ]:
test_probs, _ = forward(X_test, params)
test_pred = test_probs.argmax(axis=1)
cm = np.zeros((4, 4), dtype=int)
for true, pred in zip(y_test, test_pred):
    cm[true, pred] += 1

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
for i in range(4):
    for j in range(4):
        ax.text(j, i, cm[i,j], ha='center', va='center', color='white' if cm[i,j] > cm.max()/2 else 'black')
ax.set_xticks(range(4), CLASS_NAMES); ax.set_yticks(range(4), CLASS_NAMES)
ax.set(xlabel='Predicted class', ylabel='True class', title=f'Confusion Matrix | Test accuracy = {(test_pred==y_test).mean():.1%}')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

## 7) ดูตัวอย่างการ Predict
กราฟแท่งแสดง probability ที่ Softmax ให้แต่ละคลาส

In [ ]:
rng = np.random.default_rng(2026)
chosen = rng.choice(len(X_test), 4, replace=False)
fig, axes = plt.subplots(4, 2, figsize=(12, 10))
for row, idx in enumerate(chosen):
    axes[row,0].plot(TIME, X_test[idx], color=CLASS_COLORS[y_test[idx]], lw=1.8)
    axes[row,0].set_title(f'True: {CLASS_NAMES[y_test[idx]]} | Predicted: {CLASS_NAMES[test_pred[idx]]}')
    axes[row,0].set_ylabel('Normalized V'); axes[row,0].grid(alpha=.25)
    axes[row,1].bar(CLASS_NAMES, test_probs[idx], color=CLASS_COLORS)
    axes[row,1].set_ylim(0, 1); axes[row,1].set_ylabel('Probability'); axes[row,1].grid(axis='y', alpha=.25)
axes[-1,0].set_xlabel('Time (s)')
plt.tight_layout(); plt.show()

## สรุป

1. ADC waveform 128 samples เป็น input ของโมเดล
2. Forward propagation สร้าง probability ของสัญญาณทั้ง 4 ชนิด
3. Cross-Entropy วัดความต่างระหว่าง prediction กับ target
4. Backpropagation ส่ง error ย้อนกลับเพื่อหา gradient ของ weight และ bias
5. Gradient Descent ปรับ parameters ทำให้โมเดลแยกสัญญาณได้ดีขึ้น

**ทดลองต่อ:** เพิ่ม harmonic distortion, spike, clipping หรือสัญญาณ fault เป็นคลาสใหม่